# ДЗ 1. Рекомендательные системы на датасете MTS Library

Некрылов Леонид

Датасет: **MTS Library** — взаимодействия пользователей с книгами в сервисе МТС Библиотека (сейчас МТС Строки) за 2018–2019 годы.

Источник: https://www.kaggle.com/datasets/sharthz23/mts-library

## Как запустить

1. Установить зависимости: `pip install -r requirements.txt`
2. Данные скачиваются автоматически в папку `data/` следующей ячейкой (публичный датасет Kaggle, авторизация не нужна). Если скачивание не сработало, можно вручную скачать архив со страницы датасета и распаковать в `data/` (нужны файлы `interactions.csv`, `items.csv`, `users.csv`).
3. Запустить все ячейки по порядку. Полный прогон на CPU (Ryzen 7, 8 потоков) занимает примерно час, большая часть времени уходит на подбор гиперпараметров ALS, FM и timeSVD++.

Все зерна зафиксированы (`SEED = 42`), параметры фильтрации и разбиения заданы в начале.

In [ ]:
import os
import time
import itertools
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from models import utils, knn, ease, als, fm, timesvdpp

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = 'data'
MIN_USER = 3
MIN_ITEM = 20
VAL_START = '2019-11-01'
TEST_START = '2019-12-01'
TEST_END = '2020-01-01'
K = 10

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

In [ ]:
def download_data(path):
    if os.path.exists(path + '/interactions.csv'):
        print('data already downloaded')
        return
    os.makedirs(path, exist_ok=True)
    url = 'https://www.kaggle.com/api/v1/datasets/download/sharthz23/mts-library'
    urllib.request.urlretrieve(url, path + '/mts.zip')
    with zipfile.ZipFile(path + '/mts.zip') as z:
        z.extractall(path)
    os.remove(path + '/mts.zip')


download_data(DATA)
inter, items, users = utils.load_data(DATA)
print('interactions', inter.shape)
print('items', items.shape)
print('users', users.shape)

## 1. Данные

Три таблицы:

* `interactions.csv` — взаимодействия: `user_id`, `item_id`, `progress` (процент прочитанного, 0–100), `rating` (оценка 1–5, есть не всегда), `start_date` (дата начала чтения, точность до дня).
* `items.csv` — книги: `id`, `title`, `genres` (список жанров через запятую), `authors`, `year` (год издания).
* `users.csv` — пользователи: `user_id`, `age` (возрастная группа), `sex` (0/1).

Одно взаимодействие = пользователь открыл книгу и начал её читать. Задача — рекомендовать пользователю 10 книг, которые он ещё не читал. **Релевантным считаем объект, который пользователь начал читать в тестовом периоде** (любое новое взаимодействие). Прогресс и оценку в критерий релевантности не включаем: прогресс у книг, начатых в конце периода, ещё не успел накопиться, а оценки есть только у ~19% взаимодействий. Критерий один и тот же для всех моделей, независимо от того, на каком сигнале они обучаются.

In [ ]:
inter.head()

In [ ]:
items.head()

In [ ]:
users.head()

## 2. EDA

### Пропуски и дубликаты

In [ ]:
print('пропуски interactions')
print(inter.isna().mean().round(3))
print('пропуски items')
print(items.isna().mean().round(3))
print('пропуски users')
print(users.isna().mean().round(3))
print()
print('полные дубликаты строк:', inter.duplicated().sum())
print('дубликаты пар user-item:', inter.duplicated(['user_id', 'item_id']).sum())
print('пользователей без анкеты:', (~inter['user_id'].drop_duplicates().isin(users['user_id'])).sum())
print('книг без метаданных:', (~inter['item_id'].drop_duplicates().isin(items['id'])).sum())

In [ ]:
print(inter[inter.duplicated(['user_id', 'item_id'], keep=False)].sort_values(['user_id', 'item_id']).head(6))

Пропусков в id и датах нет. `rating` пропущен в 81% случаев — пользователи редко ставят оценки. У книг иногда нет авторов (12%) и года (22%), у пользователей иногда нет пола. Около 9 тыс. пользователей из взаимодействий отсутствуют в `users.csv` — для них демографические признаки будут пустыми.

80 повторяющихся пар user-item — повторное открытие одной книги. Оставляем одну запись с наибольшим прогрессом.

### Оценки, прогресс и время

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
inter['rating'].value_counts().sort_index().plot.bar(ax=ax[0])
ax[0].set_title('распределение оценок (без пропусков)')
inter['progress'].hist(bins=50, ax=ax[1])
ax[1].set_title('progress')
pd.to_datetime(inter['start_date']).dt.to_period('M').value_counts().sort_index().plot(ax=ax[2])
ax[2].set_title('взаимодействий в месяц')
plt.tight_layout()
plt.show()
print(inter['rating'].value_counts(normalize=True).sort_index().round(3))
print('даты:', inter['start_date'].min(), '-', inter['start_date'].max())

Оценки сильно смещены к 5 (75% всех оценок). Прогресс бимодальный: много книг брошено почти сразу, много дочитано. Взаимодействия равномерно распределены по времени с января 2018 по декабрь 2019 (около 2 тыс. в день), сезонности или резких скачков нет, так что временное разбиение по последним месяцам корректно.

### Активность пользователей и популярность книг

In [ ]:
user_cnt = inter.groupby('user_id').size()
item_cnt = inter.groupby('item_id').size()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(user_cnt, bins=np.logspace(0, 3.5, 40))
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_title('взаимодействий на пользователя')
ax[1].hist(item_cnt, bins=np.logspace(0, 4, 40))
ax[1].set_xscale('log')
ax[1].set_yscale('log')
ax[1].set_title('взаимодействий на книгу')
s = np.sort(item_cnt.values)[::-1]
ax[2].plot(np.arange(len(s)) / len(s), np.cumsum(s) / s.sum())
ax[2].set_title('доля взаимодействий vs доля самых популярных книг')
plt.tight_layout()
plt.show()
print(user_cnt.describe().round(1))
print(item_cnt.describe().round(1))
print('sparsity: %.6f' % (1 - len(inter) / (inter['user_id'].nunique() * inter['item_id'].nunique())))

У типичного пользователя 7 взаимодействий, у книги — 16. Распределения с тяжелыми хвостами: есть пользователи с тысячами книг и книги с тысячами читателей. Минимум взаимодействий у книги в исходных данных равен 2, видимо датасет уже был отфильтрован авторами. Матрица очень разреженная (99.98% нулей).

### Метаданные

In [ ]:
genres = items['genres'].dropna().str.split(',').explode()
authors = items['authors'].dropna().str.split(',').explode()
years = pd.to_numeric(items['year'].astype(str).str[:4], errors='coerce')
print('уникальных жанров:', genres.nunique(), ' жанров у книги в среднем:', round(items['genres'].dropna().str.split(',').apply(len).mean(), 2))
print('уникальных авторов:', authors.nunique())
print('книг с непонятным годом:', years.isna().sum())

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
genres.value_counts().head(20)[::-1].plot.barh(ax=ax[0])
ax[0].set_title('топ-20 жанров (по числу книг)')
years[(years > 1800) & (years < 2021)].hist(bins=50, ax=ax[1])
ax[1].set_title('год издания')
users['age'].value_counts().sort_index().plot.bar(ax=ax[2])
ax[2].set_title('возраст пользователей')
plt.tight_layout()
plt.show()
print(users['sex'].value_counts(dropna=False))

In [ ]:
tmp = inter.merge(items[['id', 'genres']], left_on='item_id', right_on='id')
tmp = tmp.assign(genre=tmp['genres'].str.split(',')).explode('genre')
g = tmp.groupby('genre').agg(n=('user_id', 'size'), progress=('progress', 'mean'), rating=('rating', 'mean'))
print(g.sort_values('n', ascending=False).head(15).round(2))
del tmp

Жанров 640, у книги в среднем ~5 жанров, авторов около 19 тыс. Каталог в основном современный (большинство книг издано после 2000), но есть и классика. Пользователи — в основном молодёжь 18–34. Жанры заметно различаются по популярности и по среднему прогрессу (например, газеты и журналы читают по чуть-чуть), так что жанр — содержательный признак для FM.

### Очистка и подвыборка

1. Удаляем дубликаты пар user-item (оставляем запись с максимальным прогрессом).
2. Итеративный k-core фильтр: оставляем книги, у которых не меньше 20 взаимодействий, и пользователей, у которых не меньше 3, пока условия не выполнятся одновременно.

Зачем: в полном каталоге 60 тыс. книг, для EASE это матрица 60k×60k (~29 ГБ во float64), что не помещается в память. Порог 20 оставляет ~13 тыс. книг и ~48% взаимодействий. Пользователи с 1–2 взаимодействиями почти не дают сигнала для коллаборативных методов. Фильтр детерминирован, так что подвыборка воспроизводится.

In [ ]:
df = utils.drop_dups(inter)
df = utils.filter_core(df, MIN_USER, MIN_ITEM)
df, user_ids, item_ids = utils.encode_ids(df)
n_users, n_items = len(user_ids), len(item_ids)
print('было:  %d взаимодействий, %d пользователей, %d книг' % (len(inter), inter['user_id'].nunique(), inter['item_id'].nunique()))
print('стало: %d взаимодействий, %d пользователей, %d книг' % (len(df), n_users, n_items))
print('оценок осталось:', df['rating'].notna().sum())
print('sparsity: %.5f' % (1 - len(df) / (n_users * n_items)))

## 3. Протокол оценки

* **Разбиение по времени** (глобальное, по дате `start_date`):
  * для подбора гиперпараметров: train = всё до 2019-11-01, validation = ноябрь 2019;
  * для финального сравнения: train = всё до 2019-12-01 (train + validation), test = декабрь 2019.
* **Пользователи для оценки**: те, у кого есть история в train и хотя бы одно новое взаимодействие в целевом месяце. Новых пользователей (без истории) модели на основе истории оценить не могут, их мало (около 300 в месяц), исключаем из оценки — для них разумно показывать популярное.
* **Кандидаты**: все книги каталога, встречавшиеся в train. Новых книг в целевых месяцах после фильтрации нет.
* **Исключение известного**: из рекомендаций убираем книги, которые пользователь уже открывал в train. Целевые объекты — только новые для пользователя книги.
* **Метрики при K=10**. Основная (выбрана до подбора): **NDCG@10** — учитывает и попадание, и позицию в списке, а пользователь видит в первую очередь верх ленты. Дополнительно: **Recall@10** (какая доля книг, которые пользователь реально начал читать, попала в топ-10) и **HitRate@10** (доля пользователей, которым хоть что-то подошло). Ещё смотрим **Coverage@10** — какая доля каталога вообще попадает в рекомендации, это показывает, не скатывается ли модель в популярное.

Все модели оцениваются на одних и тех же пользователях, кандидатах и целевых множествах.

In [ ]:
def get_day(date):
    return (pd.Timestamp(date) - pd.Timestamp('2018-01-01')).days


train_val, val = utils.time_split(df, VAL_START, TEST_START)
train_full, test = utils.time_split(df, TEST_START, TEST_END)
val_users, val_targets = utils.get_targets(train_val, val)
test_users, test_targets = utils.get_targets(train_full, test)
VAL_DAY = get_day(VAL_START)
TEST_DAY = get_day(TEST_START)

for name, tr, te, us, tg in [('val', train_val, val, val_users, val_targets), ('test', train_full, test, test_users, test_targets)]:
    print(name)
    print('  train:', len(tr), ' target month:', len(te))
    print('  пользователей для оценки:', len(us), ' целевых пар:', sum(len(v) for v in tg.values()))
    print('  новых пользователей в месяце:', te.loc[~te['uid'].isin(tr['uid']), 'uid'].nunique())
    print('  новых книг в месяце:', (~te['iid'].isin(tr['iid'])).sum())
    print('  среднее число целевых книг на пользователя: %.2f' % np.mean([len(v) for v in tg.values()]))

## 4. Модели

Общая схема: `run(name, params, train, users, day)` строит матрицу user×item по train, обучает модель, считает скоры для пользователей, убирает уже прочитанное и возвращает топ-10. Дальше `tune` перебирает сетку на validation и сортирует по NDCG@10. Для сравнения добавлен бейзлайн Popular (топ самых популярных книг в train).

Признаки для FM строятся заранее (они статичны), см. раздел про FM.

In [ ]:
fm_feats = fm.make_features(items, users, item_ids, user_ids, use_meta=True)
fm_feats_ids = fm.make_features(items, users, item_ids, user_ids, use_meta=False)
print('признаков FM с метаданными:', fm_feats['n_feat'], ' только id:', fm_feats_ids['n_feat'])


def run(name, params, train, users_eval, day):
    X = utils.make_matrix(train, n_users, n_items)
    t = time.time()
    if name == 'pop':
        pop = np.asarray(X.sum(0)).ravel()
        score_fn = lambda u: np.tile(pop, (len(u), 1))
    elif name == 'knn':
        W = knn.fit(X, **params)
        score_fn = lambda u: knn.get_scores(W, X, u)
    elif name == 'als':
        model = als.fit(X, **params)
        score_fn = lambda u: als.get_scores(model, u)
    elif name == 'ease':
        B = ease.fit(X, **params)
        score_fn = lambda u: ease.get_scores(B, X, u)
    elif name == 'fm':
        p = dict(params)
        feats = fm_feats if p.pop('use_meta', True) else fm_feats_ids
        model = fm.fit(train, feats, **p)
        score_fn = lambda u: fm.get_scores(model, u)
    elif name == 'tsvd':
        model = timesvdpp.fit(train, n_users, n_items, **params)
        score_fn = lambda u: timesvdpp.get_scores(model, u, day)
    recs = utils.recommend(score_fn, users_eval, X, k=K)
    return recs, time.time() - t


def tune(name, grid, fixed={}):
    keys = list(grid.keys())
    rows = []
    for vals in itertools.product(*[grid[k] for k in keys]):
        params = dict(zip(keys, vals))
        params.update(fixed)
        recs, sec = run(name, params, train_val, val_users, VAL_DAY)
        m = utils.calc_metrics(recs, val_users, val_targets, n_items)
        rows.append({**params, **m, 'time': round(sec, 1)})
        print(name, params, 'ndcg@10 = %.4f' % m['ndcg@10'], '(%.0f s)' % sec)
    return pd.DataFrame(rows).sort_values('ndcg@10', ascending=False).reset_index(drop=True)


def get_best(res, keys):
    best = {}
    for k in keys:
        v = res.loc[0, k]
        best[k] = v.item() if hasattr(v, 'item') else v
    return best


best_params = {}
val_results = {}

In [ ]:
recs, sec = run('pop', {}, train_val, val_users, VAL_DAY)
m = utils.calc_metrics(recs, val_users, val_targets, n_items)
val_results['Popular'] = m
best_params['Popular'] = ('pop', {})
print(m)

### 4.1 Item-based kNN

Матрица $X$ бинарная (1 — пользователь открывал книгу). Сходство книг — косинус со сжатием (shrinkage) $h$, который штрафует пары с малым числом общих читателей:

$$s_{ij} = \frac{|U_i \cap U_j|}{\sqrt{|U_i|\,|U_j|} + h}$$

Для каждой книги $i$ оставляем $k$ ближайших соседей $N_k(i)$. Скор кандидата:

$$\hat r_{ui} = \sum_{j \in I_u,\; j \in N_k(i)} s_{ji}$$

Сетка: $k \in \{20, 50, 100, 200, 500\}$, $h \in \{0, 10, 50\}$.

In [ ]:
res_knn = tune('knn', {'k': [20, 50, 100, 200, 500], 'shrink': [0, 10, 50]})
best_params['ItemKNN'] = ('knn', get_best(res_knn, ['k', 'shrink']))
val_results['ItemKNN'] = res_knn.loc[0, ['ndcg@10', 'recall@10', 'hitrate@10', 'coverage@10']].to_dict()
res_knn

In [ ]:
for sh in [0, 10, 50]:
    r = res_knn[res_knn['shrink'] == sh].sort_values('k')
    plt.plot(r['k'], r['ndcg@10'], marker='o', label='shrink=%d' % sh)
plt.xscale('log')
plt.xlabel('k')
plt.ylabel('val NDCG@10')
plt.legend()
plt.title('ItemKNN')
plt.show()

### 4.2 iALS (implicit ALS, Hu, Koren, Volinsky 2008)

Данные неявные (факт открытия книги), поэтому берём ALS для неявной обратной связи. Предпочтение $p_{ui} = 1$, если взаимодействие было, иначе 0; уверенность $c_{ui} = 1 + \alpha r_{ui}$, где $r_{ui} \in \{0,1\}$.

$$\min_{x, y} \sum_{u,i} c_{ui} (p_{ui} - x_u^T y_i)^2 + \lambda \Big(\sum_u \|x_u\|^2 + \sum_i \|y_i\|^2\Big)$$

Шаг по пользователям при фиксированных $Y$ решается точно:

$$x_u = \big(Y^TY + Y^T(C_u - I)Y + \lambda I\big)^{-1} Y^T C_u p_u$$

($Y^TY$ считается один раз, вторая часть — только по прочитанным книгам), аналогично для книг. Скор $\hat r_{ui} = x_u^T y_i$.

Сетка в два шага (чтобы не перебирать полный куб): сначала $\alpha \in \{3, 10, 30\}$ × $\lambda \in \{0.1, 1, 10\}$ при 64 факторах и 8 итерациях, затем число факторов $\{32, 64, 128\}$ при лучших $\alpha, \lambda$.

In [ ]:
res_als = tune('als', {'alpha': [3.0, 10.0, 30.0], 'reg': [0.1, 1.0, 10.0]}, fixed={'factors': 64, 'iters': 8})
res_als

In [ ]:
b = get_best(res_als, ['alpha', 'reg'])
res_als2 = tune('als', {'factors': [32, 128]}, fixed={'alpha': b['alpha'], 'reg': b['reg'], 'iters': 8})
res_als_all = pd.concat([res_als, res_als2]).sort_values('ndcg@10', ascending=False).reset_index(drop=True)
best_params['iALS'] = ('als', get_best(res_als_all, ['factors', 'alpha', 'reg', 'iters']))
val_results['iALS'] = res_als_all.loc[0, ['ndcg@10', 'recall@10', 'hitrate@10', 'coverage@10']].to_dict()
res_als_all

### 4.3 EASE (Steck 2019)

Линейная item-item модель без скрытых факторов: восстанавливаем $X$ через $XB$ с запретом на тривиальное решение $B = I$:

$$\min_B \|X - XB\|_F^2 + \lambda \|B\|_F^2, \quad \text{diag}(B) = 0$$

Решение в замкнутой форме: $P = (X^TX + \lambda I)^{-1}$, $B_{ij} = -P_{ij} / P_{jj}$ при $i \ne j$, $B_{jj} = 0$. Скор $\hat r_u = x_u B$.

Сетка: $\lambda \in \{100, 250, 500, 1000, 2000, 5000\}$.

In [ ]:
res_ease = tune('ease', {'lam': [100, 250, 500, 1000, 2000, 5000]})
best_params['EASE'] = ('ease', get_best(res_ease, ['lam']))
val_results['EASE'] = res_ease.loc[0, ['ndcg@10', 'recall@10', 'hitrate@10', 'coverage@10']].to_dict()
r = res_ease.sort_values('lam')
plt.plot(r['lam'], r['ndcg@10'], marker='o')
plt.xscale('log')
plt.xlabel('lambda')
plt.ylabel('val NDCG@10')
plt.title('EASE')
plt.show()
res_ease

### 4.4 Факторизационная машина с метапризнаками

Вектор $x$ — объединение признаков пользователя и книги:

* пользователь: id, возрастная группа (`age`, 6 значений), пол (`sex`);
* книга: id, жанры (multi-hot, веса $1/n_{genres}$), авторы (multi-hot, только авторы с $\ge 2$ книгами в каталоге, веса $1/n_{authors}$), группа года издания (до 1900, 1900–1990, 1990–2005, 2005–2012, 2012–2016, 2016+, неизвестно).

Почему эти признаки: жанр и автор — главное, по чему люди выбирают книги, и они помогают связать редкие книги с похожими популярными. Год отделяет классику от новинок. Возраст и пол влияют на жанровые предпочтения (детские книги, любовные романы и т.д.). Все признаки известны в момент рекомендации: метаданные книги статичны, анкета пользователя заполняется при регистрации. Никаких признаков из будущего (прогресс и оценки целевых книг, популярность после даты разбиения) не используется.

Модель (Rendle 2010):

$$\hat y(x) = w_0 + \sum_i w_i x_i + \sum_{i<j} \langle v_i, v_j \rangle x_i x_j$$

парное слагаемое считается за линейное время: $\frac12 \sum_f \big[(\sum_i v_{if} x_i)^2 - \sum_i v_{if}^2 x_i^2\big]$.

Обучение — BPR на неявных данных: для пары (пользователь, прочитанная книга $i$) сэмплируем случайную книгу $j$ и минимизируем

$$-\ln \sigma(\hat y(u, i) - \hat y(u, j)) + \lambda \|\cdot\|^2$$

Оптимизатор Adam (lr = 0.003, батч 8192), 8 эпох. При ранжировании слагаемые, зависящие только от пользователя, на порядок не влияют, а части книг считаются один раз для всего каталога.

Сетка: размерность $\{32, 64\}$ × $\lambda \in \{10^{-5}, 10^{-4}, 10^{-3}\}$. Чтобы понять вклад метаданных, лучшая конфигурация дополнительно обучается только на id пользователя и книги.

In [ ]:
res_fm = tune('fm', {'dim': [32, 64], 'reg': [1e-5, 1e-4, 1e-3]}, fixed={'epochs': 8})
best_params['FM'] = ('fm', get_best(res_fm, ['dim', 'reg', 'epochs']))
val_results['FM'] = res_fm.loc[0, ['ndcg@10', 'recall@10', 'hitrate@10', 'coverage@10']].to_dict()
res_fm

In [ ]:
p = dict(best_params['FM'][1])
p['use_meta'] = False
best_params['FM (только id)'] = ('fm', p)
recs, sec = run('fm', p, train_val, val_users, VAL_DAY)
val_results['FM (только id)'] = utils.calc_metrics(recs, val_users, val_targets, n_items)
print(val_results['FM'])
print(val_results['FM (только id)'])

### 4.5 timeSVD++ (Koren 2009)

Модель предсказывает явную оценку с учётом времени. Используем 1–5 звёзд там, где они есть; неявный фидбек $N(u)$ — все книги, которые пользователь открывал в train (включая неоценённые), это как раз то, ради чего в SVD++ есть слагаемое с $y_j$:

$$\hat r_{ui}(t) = \mu + b_i + b_{i,\text{Bin}(t)} + b_u + \alpha_u \,\text{dev}_u(t) + b_{u,t} + q_i^T\Big(p_u + \alpha^p_u\, \text{dev}_u(t) + |N(u)|^{-1/2}\sum_{j\in N(u)} y_j\Big)$$

* $b_{i,\text{Bin}(t)}$ — дрейф популярности книги, 30 временных корзин по ~24 дня;
* $\text{dev}_u(t) = \text{sign}(t - t_u)\,|t - t_u|^{0.4}$, $t_u$ — средний день оценок пользователя; $\alpha_u$ и $\alpha^p_u$ — линейный дрейф смещения и вкуса пользователя;
* $b_{u,t}$ — смещение пользователя в конкретный день (для будущих дней неизвестно, при предсказании = 0).

Обучаем MSE + L2 на всех параметрах, которые участвуют в примере, оптимизатор Adam (lr = 0.003). Рекомендации — топ-10 по предсказанной оценке на первый день целевого месяца. Гиперпараметры выбираем по NDCG@10 на validation (а не по RMSE), так как задача — ранжирование.

Сетка: размерность $\{16, 64\}$ × $\lambda \in \{0.01, 0.1\}$ × эпохи $\{5, 15\}$. Для оценки вклада временных компонент лучшая конфигурация также обучается без них (обычный SVD++).

In [ ]:
res_tsvd = tune('tsvd', {'dim': [16, 64], 'reg': [0.01, 0.1], 'epochs': [5, 15]})
best_params['timeSVD++'] = ('tsvd', get_best(res_tsvd, ['dim', 'reg', 'epochs']))
val_results['timeSVD++'] = res_tsvd.loc[0, ['ndcg@10', 'recall@10', 'hitrate@10', 'coverage@10']].to_dict()
res_tsvd

In [ ]:
val_r = val[val['rating'].notna() & val['uid'].isin(train_val['uid'])]
rmse = {}
for label, use_time in [('timeSVD++', True), ('SVD++ (без времени)', False)]:
    p = dict(best_params['timeSVD++'][1])
    p['use_time'] = use_time
    model = timesvdpp.fit(train_val, n_users, n_items, **p)
    pred = timesvdpp.predict_ratings(model, val_r)
    rmse[label] = np.sqrt(np.mean((pred - val_r['rating'].values) ** 2))
    if not use_time:
        best_params[label] = ('tsvd', p)
        X = utils.make_matrix(train_val, n_users, n_items)
        recs = utils.recommend(lambda u: timesvdpp.get_scores(model, u, VAL_DAY), val_users, X, k=K)
        val_results[label] = utils.calc_metrics(recs, val_users, val_targets, n_items)
rmse['константа mu'] = np.sqrt(np.mean((train_val['rating'].mean() - val_r['rating'].values) ** 2))
print('RMSE на оценках validation:', {k: round(v, 4) for k, v in rmse.items()})
print(val_results['timeSVD++'])
print(val_results['SVD++ (без времени)'])

## 5. Финальное сравнение на test

Каждая модель с выбранными гиперпараметрами переобучается на train + validation (всё до 2019-12-01) и оценивается на декабре 2019.

In [ ]:
rows = []
test_recs = {}
for label, (name, params) in best_params.items():
    recs, sec = run(name, params, train_full, test_users, TEST_DAY)
    test_recs[label] = recs
    m = utils.calc_metrics(recs, test_users, test_targets, n_items)
    rows.append({'model': label, **{k: round(v, 4) for k, v in m.items()},
                 'val ndcg@10': round(val_results[label]['ndcg@10'], 4),
                 'params': params, 'time, s': round(sec, 1)})
    print(label, m)
final = pd.DataFrame(rows).sort_values('ndcg@10', ascending=False).reset_index(drop=True)
final

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, metric in zip(ax, ['ndcg@10', 'recall@10', 'coverage@10']):
    f = final.sort_values(metric)
    a.barh(f['model'], f[metric])
    a.set_title('test ' + metric)
plt.tight_layout()
plt.show()

In [ ]:
titles = items.set_index('id')['title']
u = test_users[5]
hist = train_full[train_full['uid'] == u].sort_values('day')
print('история пользователя (последние 10):')
print(titles.reindex(item_ids[hist['iid'].values[-10:]]).values)
print('прочитал в декабре:')
print(titles.reindex(item_ids[list(test_targets[u])]).values)
for label in ['EASE', 'FM', 'timeSVD++']:
    print(label, titles.reindex(item_ids[test_recs[label][5]]).values[:5])

## 6. Выводы

CONCLUSIONS_PLACEHOLDER